# S42. Teste de Média — Bicaudal
## Two-Tailed Test for Mean

[◀ Anterior](S41_Teste_Media_Unilateral_Esquerda.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S00_Index_Estatistica.ipynb)

## 🎯 Objetivos de Aprendizagem

- Compreender o teste T bicaudal para média — o teste mais comum na prática
- Formular H₀: μ = μ₀ e H₁: μ ≠ μ₀
- Identificar a região de rejeição em ambas as caudas na distribuição t
- Implementar o two-tailed t-test com scipy.stats.ttest_1samp
- Interpretar corretamente os resultados e o intervalo de confiança

## 📝 Introdução

O **teste T bicaudal** (two-tailed t-test) é o teste mais utilizado na estatística inferencial para comparar a média de uma população com um valor de referência. Ele verifica se a **média populacional é diferente** de μ₀, sem especificar se é maior ou menor.

Situações típicas:
- "A temperatura média do processo está diferente do valor de referência?"
- "O QI médio dos alunos desta escola é igual à média nacional?"
- "O novo método de ensino produziu resultados diferentes do método tradicional?"

Por ser o teste padrão e mais aceito, é a escolha recomendada na maioria das situações.

## 📚 Explicação Detalhada

### 1. Formulação do Teste

- **H₀**: μ = μ₀ (a média é igual ao valor de referência)
- **H₁**: μ ≠ μ₀ (a média é **diferente** do valor de referência)

A região de rejeição está dividida em **ambas as caudas** da distribuição t (α/2 em cada cauda).

### 2. Estatística t e Valor Crítico

t = (x̄ - μ₀) / (s / √n), gl = n - 1

Rejeitamos H₀ se:
- **|t| > t_{α/2, gl}** (região crítica)
- **p-valor < α** (p-valor bicaudal fornecido pelo scipy)

### 3. Relação com Intervalo de Confiança

O teste T bicaudal está diretamente relacionado ao intervalo de confiança (IC) para a média:
- Se μ₀ **não estiver** dentro do IC de (1-α)×100%, rejeitamos H₀.
- Se μ₀ **estiver** dentro do IC, não rejeitamos H₀.

### 4. Teste T vs Teste Z para Média

| Característica | Teste Z | Teste T |
|---------------|---------|---------|
| σ | Conhecido | Desconhecido (estimado por s) |
| Distribuição | Normal | t de Student |
| Quando usar | Raro na prática | Quase sempre (padrão) |
| scipy | — | ttest_1samp |

## 💻 Exemplos Práticos

In [ ]:
# Importando bibliotecas
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# Exemplo 1: Teste T bicaudal completo
# Uma empresa de sucos afirma que cada embalagem contem 300ml.
# Coletamos 20 embalagens e medimos o volume.

volumes = np.array([298, 302, 297, 301, 299, 300, 296, 303, 298, 301,
                    300, 297, 302, 299, 301, 298, 300, 299, 302, 297])
mu_0 = 300.0
alpha = 0.05

# Calculo manual
n = len(volumes)
media = np.mean(volumes)
desvio = np.std(volumes, ddof=1)
gl = n - 1
erro = desvio / np.sqrt(n)
t = (media - mu_0) / erro
p_manual = 2 * (1 - stats.t.cdf(abs(t), df=gl))
t_crit = stats.t.ppf(1 - alpha/2, df=gl)

print("=== Cálculo Manual ===")
print(f"Volumes: {volumes}")
print(f"n = {n}, média = {media:.3f}, desvio = {desvio:.3f}")
print(f"H₀: μ = {mu_0} | H₁: μ ≠ {mu_0}")
print(f"t = {t:.4f} (gl = {gl})")
print(f"t crítico = ±{t_crit:.4f}")
print(f"p-valor = {p_manual:.4f}")
if abs(t) > t_crit:
    print("✅ Rejeitamos H₀: o volume médio é diferente de 300ml.")
else:
    print("❌ Não rejeitamos H₀: volume médio é compatível com 300ml.")
print()

# Usando scipy
t_scipy, p_scipy = stats.ttest_1samp(volumes, mu_0)
print("=== scipy.stats.ttest_1samp ===")
print(f"t = {t_scipy:.4f}, p-valor = {p_scipy:.6f}")
if p_scipy < alpha:
    print(f"✅ p-valor {p_scipy:.4f} < α = {alpha} → Rejeitamos H₀")
else:
    print(f"❌ p-valor {p_scipy:.4f} ≥ α = {alpha} → Não rejeitamos H₀")

In [ ]:
# Exemplo 2: Visualizacao do teste bicaudal (t)
x = np.linspace(-4, 4, 1000)
y = stats.t.pdf(x, df=gl)

plt.figure(figsize=(10, 6))
plt.plot(x, y, 'b-', label=f'Distribuição t (gl = {gl})')

# Regioes de rejeicao
x_left = np.linspace(-4, -t_crit, 100)
x_right = np.linspace(t_crit, 4, 100)
plt.fill_between(x_left, stats.t.pdf(x_left, df=gl), color='red', alpha=0.3)
plt.fill_between(x_right, stats.t.pdf(x_right, df=gl), color='red', alpha=0.3, label='Região de Rejeição')

plt.axvline(t, color='green', linestyle='--', lw=2, label=f't = {t:.2f}')
plt.axvline(-t_crit, color='red', linestyle=':', lw=2, label=f't crítico = ±{t_crit:.2f}')
plt.axvline(t_crit, color='red', linestyle=':', lw=2)
plt.title('Teste T Bicaudal')
plt.xlabel('t')
plt.ylabel('Densidade')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Exemplo 3: Relacao com intervalo de confianca
import scipy.stats as st

# IC de 95% para a media
ic_inf = media - t_crit * erro
ic_sup = media + t_crit * erro

print(f"Média amostral: {media:.3f}")
print(f"Erro padrão: {erro:.4f}")
print(f"Intervalo de Confiança (95%): [{ic_inf:.3f}, {ic_sup:.3f}]")
print(f"H₀: μ = {mu_0}")
print()
if ic_inf <= mu_0 <= ic_sup:
    print(f"μ₀ = {mu_0} está DENTRO do IC → Não rejeitamos H₀")
else:
    print(f"μ₀ = {mu_0} está FORA do IC → Rejeitamos H₀")
print()
print("O IC que contém μ₀ é equivalente a não rejeitar H₀ a 5%.")

In [ ]:
# Exemplo 4: Funcao completa para teste T bicaudal
def ttest_bicaudal(dados, mu_0, alpha=0.05):
    """Teste T bicaudal para media. Retorna estatisticas e interpretacao."""
    n = len(dados)
    media = np.mean(dados)
    desvio = np.std(dados, ddof=1)
    gl = n - 1
    erro = desvio / np.sqrt(n)
    t_stat = (media - mu_0) / erro
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=gl))
    t_crit = stats.t.ppf(1 - alpha/2, df=gl)
    ic = (media - t_crit * erro, media + t_crit * erro)
    
    print(f"=== Teste T Bicaudal para Média ===")
    print(f"Dados: n = {n}, média = {media:.3f}, desvio = {desvio:.3f}")
    print(f"H₀: μ = {mu_0} | H₁: μ ≠ {mu_0}")
    print(f"t = {t_stat:.4f} (gl = {gl})")
    print(f"Valor crítico: ±{t_crit:.4f} (α = {alpha})")
    print(f"p-valor = {p_val:.4f}")
    print(f"IC {1-alpha:.0%}: [{ic[0]:.3f}, {ic[1]:.3f}]")
    if abs(t_stat) > t_crit:
        print(f"✅ Rejeitamos H₀: μ ≠ {mu_0}")
    else:
        print(f"❌ Não rejeitamos H₀: μ = {mu_0} (não podemos afirmar diferença)")
    return {"t": t_stat, "p": p_val, "gl": gl, "ic": ic}

ttest_bicaudal(volumes, 300)
print()
ttest_bicaudal(np.array([75, 78, 72, 76, 74, 77, 73, 75, 76, 74]), 70)

In [ ]:
# Exemplo 5: Efeito do tamanho amostral
# Quanto maior n, mais facil detectar diferencas pequenas

np.random.seed(42)
print("Efeito do tamanho amostral (μ real = 102, H₀: μ = 100, σ ≈ 10):")
print()
for n in [10, 20, 50, 100, 500]:
    amostra = np.random.normal(loc=102, scale=10, size=n)
    t_s, p_s = stats.ttest_1samp(amostra, 100)
    print(f"  n = {n:3d} | média = {np.mean(amostra):.2f} | t = {t_s:.4f} | p = {p_s:.4f}", end="")
    if p_s < 0.05:
        print(" ✅ Rejeita")
    else:
        print(" ❌ Não rejeita")

In [ ]:
# Exemplo 6: Comparacao bicaudal vs unilateral para media
print("COMPARAÇÃO: Bicaudal vs Unilateral (Teste T)")
print("=" * 45)
print()

dados_teste = np.array([102, 105, 98, 101, 104, 99, 103, 100, 106, 97])
t_s, p_bi = stats.ttest_1samp(dados_teste, 100)
p_esq = p_bi / 2 if t_s < 0 else 1 - p_bi / 2
p_dir = p_bi / 2 if t_s > 0 else 1 - p_bi / 2

print(f"Média amostral: {np.mean(dados_teste):.2f}")
print(f"H₀: μ = 100")
print(f"t = {t_s:.4f}")
print()
print(f"p-valor (bicaudal):        {p_bi:.4f}")
print(f"p-valor (unilateral esq.): {p_esq:.4f}")
print(f"p-valor (unilateral dir.): {p_dir:.4f}")

## ✅ Exercícios Resolvidos

**Exercício 1:** Uma máquina deve produzir parafusos com comprimento médio de 5cm. Uma amostra de 16 parafusos tem comprimentos: [4.98, 5.02, 4.97, 5.01, 4.99, 5.00, 4.96, 5.03, 4.98, 5.01, 5.00, 4.97, 5.02, 4.99, 5.01, 4.97]. Teste a 5%.

In [ ]:
# Solucao do Exercicio 1
parafusos = np.array([4.98, 5.02, 4.97, 5.01, 4.99, 5.00, 4.96, 5.03,
                      4.98, 5.01, 5.00, 4.97, 5.02, 4.99, 5.01, 4.97])
t_stat, p_val = stats.ttest_1samp(parafusos, 5.0)
print(f"Média: {np.mean(parafusos):.4f}")
print(f"H₀: μ = 5.0 | H₁: μ ≠ 5.0")
print(f"t = {t_stat:.4f}, p-valor = {p_val:.4f}")
if p_val < 0.05:
    print("✅ Rejeitamos H₀: comprimento médio diferente de 5cm.")
else:
    print("❌ Não rejeitamos H₀: comprimento médio = 5cm.")

**Exercício 2:** Uma turma de 30 alunos fez um teste de proficiência. A média foi 78.4 com desvio 6.3. A média nacional é 75. Teste a 1% se a turma é diferente da média nacional.

In [ ]:
# Solucao do Exercicio 2 (dados agregados)
n = 30
media = 78.4
desvio = 6.3
mu_0 = 75.0
alpha = 0.01

t = (media - mu_0) / (desvio / np.sqrt(n))
gl = n - 1
p = 2 * (1 - stats.t.cdf(abs(t), df=gl))
t_crit = stats.t.ppf(1 - alpha/2, df=gl)

print(f"H₀: μ = {mu_0} | H₁: μ ≠ {mu_0}")
print(f"t = {t:.4f} (gl = {gl})")
print(f"t crítico = ±{t_crit:.4f}")
print(f"p-valor = {p:.4f}")
if p < alpha:
    print(f"✅ Rejeitamos H₀ a {alpha}: a turma é diferente da média nacional.")
else:
    print(f"❌ Não rejeitamos H₀ a {alpha}.")

**Exercício 3:** Um estudo afirma que a pressão arterial sistólica média é 120 mmHg. Em 25 pacientes, a média é 124 com desvio 8. Teste a 5%.

In [ ]:
# Solucao do Exercicio 3
n = 25
media = 124
desvio = 8
mu_0 = 120

t = (media - mu_0) / (desvio / np.sqrt(n))
p = 2 * (1 - stats.t.cdf(abs(t), df=n-1))

print(f"H₀: μ = {mu_0} | H₁: μ ≠ {mu_0}")
print(f"t = {t:.4f}, p-valor = {p:.4f}")
if p < 0.05:
    print("✅ Rejeitamos H₀: pressão média diferente de 120.")
else:
    print("❌ Não rejeitamos H₀.")

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Teste se a média de [55, 58, 52, 57, 54, 56, 53, 55] é diferente de 56 (α = 0.05). |
| 🟡 Médio | 2. Um fabricante afirma que a vida útil média é 800h. 18 peças: média 785h, desvio 25h. Teste a 5%. |
| 🔴 Difícil | 3. Crie um script que realize o teste T bicaudal, calcule o IC de 95% e o Cohen's d, e apresente um relatório completo. |

In [ ]:
# 🟢 Exercício 1: Escreva sua solução aqui
dados = np.array([55, 58, 52, 57, 54, 56, 53, 55])
# Complete


In [ ]:
# 🟡 Exercício 2: Escreva sua solução aqui
n = 18
media = 785
desvio = 25
mu_0 = 800
# Complete


In [ ]:
# 🔴 Exercício 3: Relatorio completo
def relatorio_completo(dados, mu_0, alpha=0.05):
    # Implemente
    pass

## 🏆 Desafios

1. Explique como usar o teste T bicaudal para determinar se um processo está sob controle estatístico.
2. Pesquise sobre **teste T para duas amostras** (two-sample t-test) e crie um exemplo em Python.
3. Investigue o efeito de outliers no teste T bicaudal: crie dados com e sem outlier e compare os resultados.

In [ ]:
# Desafio 3: Efeito de outlier
# Dados limpos
dados_limpos = np.array([4.98, 5.02, 4.97, 5.01, 4.99, 5.00, 4.96, 5.03,
                         4.98, 5.01, 5.00, 4.97, 5.02, 4.99, 5.01, 4.97])
# Dados com outlier
dados_outlier = dados_limpos.copy()
dados_outlier[0] = 10.5  # outlier extremo

t_clean, p_clean = stats.ttest_1samp(dados_limpos, 5.0)
t_out, p_out = stats.ttest_1samp(dados_outlier, 5.0)

print("Efeito de Outlier no Teste T:")
print(f"  Sem outlier: t = {t_clean:.4f}, p = {p_clean:.4f}")
print(f"  Com outlier: t = {t_out:.4f}, p = {p_out:.4f}")
print(f"  p-valor {'AUMENTOU' if p_out > p_clean else 'DIMINUIU'} drasticamente!")

## 📌 Resumo

- O **teste T bicaudal** verifica se μ é **diferente** de μ₀.
- H₀: μ = μ₀, H₁: μ ≠ μ₀. Região de rejeição em ambas as caudas.
- Estatística t = (x̄ - μ₀) / (s / √n), gl = n - 1.
- O `ttest_1samp` do scipy já retorna o p-valor bicaudal.
- Existe equivalência com o intervalo de confiança: se μ₀ está fora do IC, rejeitamos H₀.
- É o teste padrão recomendado na maioria das situações práticas.

## 💡 Curiosidades

O teste T para uma amostra é a base de muitos outros testes estatísticos. O **teste T pareado**, por exemplo, nada mais é que um teste T de uma amostra aplicado às **diferenças** entre medidas pareadas. Já o **teste T para duas amostras independentes** é uma extensão que compara duas médias. Dominar o teste T de uma amostra é o primeiro passo para entender toda a família de testes T.

## ✅ Boas Práticas

1. O teste T bicaudal é o padrão — use-o a menos que haja justificativa para unilateral.
2. Verifique as suposições: normalidade (especialmente para n < 30).
3. Sempre reporte o p-valor exato, não apenas "significante" ou "não significante".
4. Inclua o intervalo de confiança da média junto com o teste.
5. Calcule o tamanho do efeito (Cohen's d) para avaliar a relevância prática.

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| Usar teste T com dados não normais e n pequeno | Use transformação ou teste não paramétrico (Wilcoxon) |
| Interpretar p > 0.05 como "H₀ é verdadeira" | p > 0.05 significa evidência insuficiente, não confirmação |
| Ignorar outliers | Outliers podem distorcer completamente o teste |
| Confundir significância estatística com prática | Um efeito pode ser significante mas irrelevante |
| Usar o desvio populacional (ddof=0) no teste T | O teste T usa o desvio amostral (ddof=1) |

## 📖 Referências

- [W3Schools - T-Test Two-Tailed](https://www.w3schools.com/statistics/statistics_ttest_two-tailed.php)
- [SciPy - ttest_1samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html)
- [OpenIntro Statistics - Inference for Means](https://www.openintro.org/book/os/)
- [StatQuest - T Tests (YouTube)](https://www.youtube.com/watch?v=pTmLQvMM-1M)

[◀ Anterior](S41_Teste_Media_Unilateral_Esquerda.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S00_Index_Estatistica.ipynb)